# 第2回 Advanced — 再現実験と検証済みソース回答の比較

このノートブックは**再実行用の改訂版**です。既存の学習と比較に加え、次を自動実行します。

1. 講座リポジトリのコミットを固定し、用語集のSHA-256を検証
2. 学習前 / LoRA後 / 元のRAGの32問比較
3. 独立して**検索（用語ID）・LLM生成・JSON形式・出典一致**を測定
4. 用語集の定義を直接返す決定的な方式と、LLMが作った回答を安全にフォールバックさせる方式を区別
5. 元ログとの混同を避けて追加評価結果を別名で保存

**実測済み**：別のColabセッションで末尾まで実行し、結果を `lectures/02/evidence/revised_rag_evaluation.json` および `natural_questions_retrieval.json` として添付済みです。配布するノートブックには出力を埋め込んでいません。再現・新条件の追加検証が必要な場合はColab T4 GPUを選び、**ランタイム → すべてのセルを実行**してください。APIキー不要です。教育課題用の実験であり、本番サービスの保証ではありません。

出力が不正な場合、用語集にある**根拠文をそのまま返す**ことを最終回答とします。これによる成功は **LLM単体の能力向上とは評価しません**。

# 第2回 Advanced｜用語集SFT・LoRA・検索比較実験

**研究課題**：用語集由来のQ&AでLoRAを学習させると、未学習の言い換え・未学習用語・用語集外質問への応答はどう変わるか。また、学習と検索の役割はどう異なるか。

**比較する3条件**：①基盤モデル（FT前・資料なし）／②LoRA後（資料なし）／③基盤モデル＋用語集検索（RAG。資料あり）。同一モデルrevision、質問、デコード設定で比較する。RAGは別の入力条件なので、**FTの因果効果は①↔②のみ**から読む。

**テスト設計**：用語ID単位で学習/保留を分割。質問表現を学習とは別にする。`seen_paraphrase`（学習用語の言い換え）、`seen_alias`（別名）、`heldout_term`（学習から除いた用語）、`unknown`（用語集にない用語）を区別する。

**評価指標**：`format_rate`=JSONスキーマ遵守、`decision_rate`=`needs_review`判定正解、`both_rate`=その両方、`content_proxy_rate`=参照定義との文字列類似度等の代理基準に合致した割合（**意味的正しさの証明ではない**）。元の演習の`content_rate`との混同を避ける。失敗例を人が評価する。

**注意**：用語集の情報は「講座の用語集に含まれる」ことと「世界的に真実である」ことが同義ではない。実際の学習実行とスコアはColab GPUで生成する。**このノートブック自体には学習済み結果は入っていない**。

## 0. 実行環境と依存関係
講座の公式ノートブックと同じ固定バージョンを使用。Google Colabの **T4 GPU** を選択して上から順に実行する。

In [ ]:
import subprocess, sys, shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi がありません。Colab → ランタイム → ランタイムのタイプを変更 → T4 GPU にしてください。")
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k,v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}

## 1. 実験設定と出典固定
用語集と`make_sft_dataset.py`は講座リポジトリから取得。GitコミットSHAと用語集のSHA256を結果に残す。

In [ ]:
from pathlib import Path
import json, hashlib, time, random, math, sys, subprocess, shutil
import pandas as pd
from difflib import SequenceMatcher
import torch, transformers, peft, trl

CONFIG = {
    "model_id": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "heldout_ratio": 0.25,
    "max_eval_per_group": 8,
    "epochs": 4,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 768,
    "max_new_tokens": 224,
    "do_sample": False,
    "work_dir": "/content/lecture02_advanced",
}
random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
if not torch.cuda.is_available():
    raise RuntimeError("学習用GPUが必要です。ColabのランタイムをT4 GPUへ切り替えてください")
WORK=Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
SRC=WORK/"course_repo"
if not SRC.exists():
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/matsuolab/lecture-ai-engineering-2026.git", str(SRC)], check=True)
# 講座repoは将来更新されるため、この実験で使ったコミットへ固定する
PINNED_COURSE_COMMIT="fae899a409e2c99feee9d44a3789f936f5fc64ee"
probe=subprocess.run(["git", "-C", str(SRC), "cat-file", "-e", PINNED_COURSE_COMMIT+"^{commit}"],capture_output=True)
if probe.returncode != 0:
    subprocess.run(["git", "-C", str(SRC), "fetch", "--depth=1", "origin", PINNED_COURSE_COMMIT],check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "--detach", PINNED_COURSE_COMMIT],check=True)
course_commit=subprocess.check_output(["git", "-C", str(SRC), "rev-parse", "HEAD"], text=True).strip()
assert course_commit==PINNED_COURSE_COMMIT
glossary_path=SRC/"data/glossary.json"
glossary_bytes=glossary_path.read_bytes()
glossary_sha256=hashlib.sha256(glossary_bytes).hexdigest()
assert glossary_sha256=="39de4f52e3de0b63cbd1500126ccdfa7863c83554ea1f9b3bf3f336c55a68e8c", "用語集が保存済み実験と異なります"
glossary=json.loads(glossary_bytes)["terms"]
script=SRC/"scripts/make_sft_dataset.py"
res=subprocess.run([sys.executable, str(script)], text=True, capture_output=True, check=True)
raw_sft=[json.loads(s) for s in res.stdout.splitlines() if s.strip()]
assert len(raw_sft)==2*len(glossary), "スクリプトの出力形式・テンプレート数が想定と違います"
print("公式用語集件数:",len(glossary),"生成Q&A:",len(raw_sft),"course commit:",course_commit)

## 2. 学習/検証/評価データの設計
講座の`make_sft_dataset.py`が出力した`messages`を使用。`needs_review`付きJSONへ教師回答を変換する。**同じ用語IDが学習集合と`heldout_term`評価集合に跨らない**。

追加した拒否例（架空用語）を学習に含めることで、`needs_review=true`の振る舞いも学習させる。ただし未知語の評価例はこれとは別。

In [ ]:
SYSTEM = ('あなたはAIE2026用語解説アシスタントです。'
          '質問された用語の意味を確実に説明できるときだけ回答してください。'
          '不確かな場合は「この用語については確認が必要です。」と回答してください。'
          '必ず1行のJSONオブジェクトのみ返してください。'
          'キーはanswer（文字列）とneeds_review（真偽値）の2つだけです。')
RAG_SYSTEM = ('次の「用語集検索結果」だけを根拠として回答してください。'
              '検索結果が「該当なし」のときは、不確かな知識で埋めず確認が必要としてください。'
              '必ずキーanswer（文字列）、needs_review（真偽値）だけの1行JSONを返してください。')
REFUSAL = "この用語については用語集に記載がありません。確認が必要です。"
make_target=lambda answer, needs: json.dumps({"answer": answer, "needs_review": needs},ensure_ascii=False)

# stable: 用語の表示順に影響されないようidで並べ、専用の乱数で分割
sorted_terms=sorted(glossary,key=lambda x:x["id"])
ids=[x["id"] for x in sorted_terms]
rr=random.Random(CONFIG["seed"]);rr.shuffle(ids)
hold_count=max(5,round(len(ids)*CONFIG["heldout_ratio"]))
heldout=set(ids[:hold_count]); seen=set(ids[hold_count:])
assert heldout.isdisjoint(seen) and seen.union(heldout)==set(ids)

# 講座スクリプトの2行/用語という出力仕様に基づき対応づける
sft_by_id={}
for i,term in enumerate(glossary):
    a,b=raw_sft[2*i:2*i+2]
    if term["term"] not in a["messages"][0]["content"] or term["term"] not in b["messages"][0]["content"]:
        raise ValueError(f"SFT出力の用語対応が一致しません: {term['id']}")
    if a["messages"][-1]["content"]!=term["definition"]:
        raise ValueError(f"SFTの回答と用語集が一致しません: {term['id']}")
    sft_by_id[term["id"]]=[a,b]

train_records=[]; valid_records=[]
for term in sorted_terms:
    if term["id"] not in seen: continue
    for record in sft_by_id[term["id"]]:
        q=record["messages"][0]["content"]
        ans=record["messages"][-1]["content"]
        train_records.append({"messages":[{"role":"system","content":SYSTEM},
            {"role":"user","content":q},
            {"role":"assistant","content":make_target(ans,False)}]})
    # 学習に含めない言い回し: パラメータ選定用(valid)。testではさらに別テンプレート
    valid_records.append({"messages":[{"role":"system","content":SYSTEM},
        {"role":"user","content":f"「{term['term']}」の役割を短く説明してください。"},
        {"role":"assistant","content":make_target(term['definition'],False)}]})

negative_train_terms=["架空の月光最適化器", "紫雲勾配正規化", "七色トークン圧縮法", "銀河報酬分解", "水晶量子リランキング", "桜花アダプタ学習"]
for unknown in negative_train_terms:
    train_records.append({"messages":[{"role":"system","content":SYSTEM},
        {"role":"user","content":f"「{unknown}」とは何ですか？"},
        {"role":"assistant","content":make_target(REFUSAL,True)}]})

# テスト用: 学習時に使われない質問テンプレート
term_by_id={x["id"]:x for x in glossary}
examples=[]
def add_case(category,question,term_id=None):
    definition=term_by_id[term_id]["definition"] if term_id else None
    examples.append({"category":category,"question":question,"term_id":term_id,
                     "expected_review":term_id is None,"reference":definition})
for term_id in sorted(seen)[:CONFIG["max_eval_per_group"]]:
    add_case("seen_paraphrase",f"「{term_by_id[term_id]['term']}」はどのような仕組み・概念を指しますか？",term_id)
for term_id in sorted(seen):
    term=term_by_id[term_id]
    aliases=[s for s in term.get("aliases",[]) if len(s.strip())>=4 and s.lower()!=term["term"].lower()]
    if aliases: add_case("seen_alias",f"「{aliases[0]}」という言い方はどういう意味ですか？",term_id)
    if sum(e["category"]=="seen_alias" for e in examples)>=CONFIG["max_eval_per_group"]: break
for term_id in sorted(heldout)[:CONFIG["max_eval_per_group"]]:
    add_case("heldout_term",f"「{term_by_id[term_id]['term']}」の意味を教えてください。",term_id)

unknown_eval_candidates=["Speculative Decoding", "KV cache", "Sparse Autoencoder", "DPO", "GRPO", "Conformal Prediction", "GQA", "RLAIF", "Blackwell GPU", "Mixture of Depths", "Fréchet Inception Distance", "FlashAttention"]
all_names={s.lower() for t in glossary for s in [t["term"],*t.get("aliases",[])]}
for name in unknown_eval_candidates:
    if name.lower() in all_names: continue
    add_case("unknown",f"「{name}」をこの講座の用語集に基づいて説明してください。")
    if sum(e["category"]=="unknown" for e in examples)>=CONFIG["max_eval_per_group"]:break
assert sum(e["category"]=="unknown" for e in examples)>=5, "用語集外の評価例が不足"
questions=[e["question"] for e in examples]
train_questions=[r["messages"][1]["content"] for r in train_records]
valid_questions=[r["messages"][1]["content"] for r in valid_records]
assert len(questions)==len(set(questions))
assert not set(questions)&(set(train_questions)|set(valid_questions)), "データ漏洩: 評価質問が学習/検証と重複"
assert not heldout & {x["id"] for x in sorted_terms if x["id"] in seen}

(WORK/"train_messages.jsonl").write_text("\n".join(json.dumps(r,ensure_ascii=False) for r in train_records)+"\n",encoding="utf-8")
(WORK/"validation_messages.jsonl").write_text("\n".join(json.dumps(r,ensure_ascii=False) for r in valid_records)+"\n",encoding="utf-8")
(WORK/"test_cases.json").write_text(json.dumps(examples,ensure_ascii=False,indent=2),encoding="utf-8")
(WORK/"split.json").write_text(json.dumps({"train_term_ids": sorted(seen),"heldout_term_ids":sorted(heldout),"negative_train_terms":negative_train_terms},ensure_ascii=False,indent=2),encoding="utf-8")
print("学習:",len(train_records),"検証:",len(valid_records),"テスト:",len(examples),"学習用語:",len(seen),"保留用語:",len(heldout))
print(pd.Series([e["category"] for e in examples]).value_counts())

## 3. 検索関数とスコアリングの検証
用語検索は文字列一致の簡易retriever。正式なembedding検索ではないため、別名の重複・未知の言い回し・質問の意味理解には弱い。評価ではこの限界も記録。

In [ ]:
import re

def retrieve(question: str):
    # この評価セットは全問が「用語」を明示する。完全一致を要求し、部分一致による誤検索を避ける。
    quoted=re.search(r"「([^」]+)」",question)
    if not quoted: return None  # 略語抽出や意味検索はこの簡易retrieverの対象外
    name=quoted.group(1).strip().casefold()
    hits=[t for t in glossary if any(name==alias.strip().casefold()
           for alias in [t["term"],*t.get("aliases",[])])]
    return hits[0] if len(hits)==1 else None  # 同名異義の曖昧さは拒否

def check_format(raw):
    try:
        obj=json.loads(raw)
    except (ValueError,TypeError):return False,"json_parse",None
    if not isinstance(obj,dict):return False,"not_object",None
    if set(obj)!={"answer","needs_review"}:return False,"keys",obj
    if type(obj["answer"]) is not str or type(obj["needs_review"]) is not bool:return False,"types",obj
    return True,"",obj

def grade(raw,case):
    fmt, reason, obj=check_format(raw)
    decision=fmt and obj["needs_review"]==case["expected_review"]
    answer=obj["answer"] if fmt else ""
    if case["reference"] is None:
        proxy=bool(decision and ("確認" in answer or "記載" in answer))
        similarity=None
    else:
        similarity=SequenceMatcher(None,answer,case["reference"]).ratio() if fmt else 0.0
        proxy=decision and similarity>=0.60
    # proxy は文字列指標であり、人手の事実正誤判定とは異なる
    return {"format_ok":fmt,"format_reason":reason,"decision_ok":bool(decision),
            "content_proxy_ok":bool(proxy),"text_similarity":similarity,
            "both_ok":bool(fmt and decision)}

assert check_format('{"answer":"x","needs_review":true}')[0]
assert not check_format('{"answer":"x","needs_review":"true"}')[0]
assert not check_format('{"answer":"x","needs_review":false,"extra":1}')[0]
assert grade('{"answer":"確認が必要です。","needs_review":true}',{"reference":None,"expected_review":True})["both_ok"]
assert not grade('{"answer":"創作","needs_review":false}',{"reference":None,"expected_review":True})["decision_ok"]
print("スコアラーの自己テスト: PASS")

## 4. 学習前モデル・検索付きモデルの評価
**前処理と質問のみに差がある条件は混ぜない**。①と②は同じプロンプトで比較。③は用語集の検索結果を追加した別システムの比較。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

tok=AutoTokenizer.from_pretrained(CONFIG["model_id"],revision=CONFIG["revision"])
if tok.pad_token is None:tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(CONFIG["model_id"],revision=CONFIG["revision"],dtype=torch.float32).to("cuda")
model.eval()
print("読み込んだモデル:",CONFIG["model_id"],"重みの使用量GB:",round(model.get_memory_footprint()/1e9,2))

def messages_for(q,mode="plain"):
    if mode=="plain":
        return [{"role":"system","content":SYSTEM},{"role":"user","content":q}]
    if mode=="rag":
        hit=retrieve(q)
        snippet=(f"用語: {hit['term']}\n定義: {hit['definition']}" if hit else "該当なし")
        return [{"role":"system","content":RAG_SYSTEM},
                {"role":"user","content":f"用語集検索結果:\n{snippet}\n\n質問: {q}"}]
    raise ValueError(mode)

def generate(model,question,mode="plain"):
    prompt=tok.apply_chat_template(messages_for(question,mode),tokenize=False,
        add_generation_prompt=True,enable_thinking=False)
    encoded=tok(prompt,return_tensors="pt").to(model.device)
    with torch.no_grad():
        generated=model.generate(**encoded,max_new_tokens=CONFIG["max_new_tokens"],
                do_sample=CONFIG["do_sample"],pad_token_id=tok.pad_token_id)
    return tok.decode(generated[0][encoded["input_ids"].shape[1]:],skip_special_tokens=True).strip()

def evaluate(model,label,mode):
    rows=[]; started=time.perf_counter()
    for j,c in enumerate(examples):
        raw=generate(model,c["question"],mode)
        rows.append({**{k:v for k,v in c.items() if k!="reference"},"raw":raw,**grade(raw,c)})
        print(f"{label}: {j+1}/{len(examples)}",end="\r")
    elapsed=round(time.perf_counter()-started,2)
    df=pd.DataFrame(rows)
    summary={"label":label,"mode":mode,"n":len(rows),"elapsed_seconds":elapsed,
             "format_rate":round(float(df.format_ok.mean()),4),
             "decision_rate":round(float(df.decision_ok.mean()),4),
             "content_proxy_rate":round(float(df.content_proxy_ok.mean()),4),
             "both_rate":round(float(df.both_ok.mean()),4),
             "by_group": {group:{"n":len(g),"format_rate":round(float(g.format_ok.mean()),4),
                        "decision_rate":round(float(g.decision_ok.mean()),4),
                        "both_rate":round(float(g.both_ok.mean()),4)} for group,g in df.groupby("category")}}
    (WORK/f"eval_{label}.json").write_text(json.dumps({"summary":summary,"rows":rows},ensure_ascii=False,indent=2),encoding="utf-8")
    print("\n",json.dumps({k:v for k,v in summary.items() if k!="by_group"},ensure_ascii=False))
    return summary,rows

before_summary,before_rows=evaluate(model,"before","plain")
rag_summary,rag_rows=evaluate(model,"rag","rag")

## 5. LoRA学習（用語集由来データ）

**学習元**：講座の`make_sft_dataset.py`によるmessages形式の生成結果。保留用語は完全除外。学習データ追加：6件の架空用語への拒否例。ベースモデルの更新はせずLoRAアダプタのみ学習。

計算量と訓練ステップ数は元の架空FAQ実験と異なるため、実測して記録する。

In [ ]:
train_ds=Dataset.from_list(train_records)
valid_ds=Dataset.from_list(valid_records)
model.config.use_cache=False
lora=LoraConfig(r=CONFIG["lora_r"],lora_alpha=CONFIG["lora_alpha"],lora_dropout=0.05,
    bias="none",task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
sft=SFTConfig(output_dir=str(WORK/"checkpoints"),num_train_epochs=CONFIG["epochs"],
     per_device_train_batch_size=4,gradient_accumulation_steps=2,
     learning_rate=CONFIG["lr"],lr_scheduler_type="cosine",warmup_steps=4,
     logging_steps=10,save_strategy="epoch",save_total_limit=2,
     max_length=CONFIG["max_seq_len"],packing=False,report_to="none",seed=CONFIG["seed"],
     assistant_only_loss=True)  # system/user入力は教師損失の対象にしない
trainer=SFTTrainer(model=model,args=sft,train_dataset=train_ds,eval_dataset=valid_ds,
                   peft_config=lora,processing_class=tok)
trainer.model.print_trainable_parameters()
train_start=time.perf_counter()
training=trainer.train()
train_seconds=round(time.perf_counter()-train_start,2)
logged_losses=[x["loss"] for x in trainer.state.log_history if "loss" in x]
last_logged_loss=logged_losses[-1] if logged_losses else None
tuned=trainer.model
tuned.config.use_cache=True
tuned.eval()
adapter_dir=WORK/"adapter"
tuned.save_pretrained(adapter_dir)
print("学習完了:",train_seconds,"seconds | steps:",trainer.state.global_step,
      "final train loss:",training.training_loss)

## 6. 学習後評価、群別比較、失敗分析
評価質問・decode設定は①と②で同一。回答内容を人が確認して、文字列類似度ベースの代理指標と区別する。

In [ ]:
after_summary,after_rows=evaluate(tuned,"after","plain")
summary_frame=pd.DataFrame([{k:v for k,v in s.items() if k!="by_group"} for s in [before_summary,after_summary,rag_summary]])
display(summary_frame)
groups=[]
for s in [before_summary,after_summary,rag_summary]:
    for group,g in s["by_group"].items(): groups.append({"condition":s["label"],"group":group,**g})
display(pd.DataFrame(groups).sort_values(["group","condition"]))
paired=[{"category":b["category"],"question":b["question"],"before_ok":b["both_ok"],
         "after_ok":a["both_ok"],"before_raw":b["raw"],"after_raw":a["raw"]}
        for b,a in zip(before_rows,after_rows)]
improved=[p for p in paired if not p["before_ok"] and p["after_ok"]]
regressed=[p for p in paired if p["before_ok"] and not p["after_ok"]]
print("改善数:",len(improved),"退行数:",len(regressed))
failures=pd.DataFrame([{k:v for k,v in r.items() if k in ["category","question","format_reason","decision_ok","content_proxy_ok","raw"]}
                        for r in after_rows if not r["both_ok"]])
print("学習後に残った失敗例（以下の出力は事実確認が必要）:")
display(failures)

## 7. 再現性メタデータ・結果出力・回答草案
`result.json`には観測値を記録する。**学習が成功したときだけ**`executed_training_myself=True`を保存する。人手で正誤を検査してから提出。

In [ ]:
metadata={
  "executed_training_myself":True,
  "method":"Advanced: glossary SFT (LoRA) vs plain vs retrieval-augmented base model",
  "model":CONFIG["model_id"], "revision":CONFIG["revision"],
  "course_commit":course_commit,"glossary_sha256":glossary_sha256,
  "config":CONFIG,"versions":{m.__name__:m.__version__ for m in [torch,transformers,peft,trl]},
  "counts":{"glossary_terms":len(glossary),"seen_terms":len(seen),"heldout_terms":len(heldout),
            "train_records":len(train_records),"valid_records":len(valid_records),"test_cases":len(examples)},
  "training":{"seconds":train_seconds,"steps":trainer.state.global_step,
              "epochs_completed":trainer.state.epoch,"mean_train_loss":training.training_loss,
              "last_logged_loss":last_logged_loss},
  "evaluations":{"before":before_summary,"after":after_summary,"rag":rag_summary},
  "changes":{"improved_count":len(improved),"regressed_count":len(regressed),
             "improved_examples":improved[:3],"regressed_examples":regressed[:3]},
  "limitations":["単一seedの探索的実験。差の有意性は主張しない",
                 "類似度は参照文との文字列一致に基づく代理指標。人手で事実性を確認する",
                 "用語IDを分けたのでheldout_termはFT学習には含まれないがRAGには検索可能",
                 "RAG条件は参照テキストを入力するため、LoRA有無だけの純粋比較ではない",
                 "グロサリーは教材ドラフトであり最新事実の保証ではない"]
}
(WORK/"advanced_result.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")

def fmt(v):return f"{100*v:.1f}%"
md=f"""# 第2回 解答

## 演習

- モデル：{CONFIG["model_id"]}（revision `{CONFIG["revision"]}`）
- 演習の元の架空FAQによる実験：**未実施**。このノートブックでは発展課題の用語集実験を実施。
- 自分で学習を実行したか：はい（このColab実行でLoRA学習を完了）

## 宿題

- 取り組んだレベル：**発展（Advanced）**
- 目的：講座の用語集からSFT用データを構築し、LoRA学習と検索併用の違いを検証する。
- 元データ：講座リポジトリ `scripts/make_sft_dataset.py` の出力。取得コミット `{course_commit}`。
- モデル：{CONFIG["model_id"]}、固定revision、LoRA r={CONFIG["lora_r"]}、alpha={CONFIG["lora_alpha"]}、学習率={CONFIG["lr"]}、{CONFIG["epochs"]} epochs、seed={CONFIG["seed"]}。
- データ：学習用語{len(seen)}件、用語単位の保留{len(heldout)}件、学習例{len(train_records)}件、テスト{len(examples)}問。うち拒否学習例6件。評価用未知語は学習と重複しない。
- 学習時間：{train_seconds:.1f}秒、step={trainer.state.global_step}、mean_train_loss={training.training_loss:.4f}、last_logged_loss={last_logged_loss}。

### 実験結果

| 条件 | format_rate | decision_rate | content_proxy_rate | both_rate | 評価時間 |
|---|---:|---:|---:|---:|---:|
| ベース（資料なし） | {fmt(before_summary["format_rate"])} | {fmt(before_summary["decision_rate"])} | {fmt(before_summary["content_proxy_rate"])} | {fmt(before_summary["both_rate"])} | {before_summary["elapsed_seconds"]}秒 |
| LoRA後（資料なし） | {fmt(after_summary["format_rate"])} | {fmt(after_summary["decision_rate"])} | {fmt(after_summary["content_proxy_rate"])} | {fmt(after_summary["both_rate"])} | {after_summary["elapsed_seconds"]}秒 |
| 検索＋ベースモデル | {fmt(rag_summary["format_rate"])} | {fmt(rag_summary["decision_rate"])} | {fmt(rag_summary["content_proxy_rate"])} | {fmt(rag_summary["both_rate"])} | {rag_summary["elapsed_seconds"]}秒 |

- ペア比較：改善{len(improved)}件、退行{len(regressed)}件。群別スコアは`advanced_result.json`を参照。
- 指標の注意：`decision_rate`は`needs_review`が正しい割合、`content_proxy_rate`は参照定義との文字列類似度などを用いた代理評価であり、真の事実正確性を保証しない。

### 失敗分析と考察（提出前に原文を確認して追記）

- 退行した質問と原因：**[eval_after.jsonの該当回答を確認して記入]**
- 学習で改善した回答の具体例：**[eval_before.json / eval_after.jsonから引用]**
- 学習していない用語に対するLoRAの回答：**[heldout_term群を確認]**
- 用語集にない質問への確認要求：**[unknown群の誤断定を確認]**
- 検索の限界：本実験のretrieverは単純な部分文字列一致のため、曖昧な別名や複数用語を含む質問で誤検索の可能性がある。検索結果を受け取った生成モデルが常に正しいとも限らない。
- FTの適用判断：出力形式・回答スタイルを安定化したい場合に適する可能性がある。一方、頻繁に変更される定義や学習対象外の知識は外部検索で参照する方が更新容易。今回の群別結果から、何がどこまで改善したかを区別する。
- 制約：単一seed、少数例、文字列代理評価。差の統計的有意性は検証していない。実運用判断には複数seedと人手の事実性評価が必要。
"""
(WORK/"answer_draft.md").write_text(md,encoding="utf-8")
print("保存先:",WORK)
print("必要ファイル: advanced_result.json / eval_before.json / eval_after.json / eval_rag.json / answer_draft.md")
print("提出前に人手で回答原文・考察・演習の実績を必ず確認してください")

## 8. ZIPにしてダウンロード
Colabのランタイムは終了すると消えることがある。結果を保存してから「ランタイムを切断して削除」する。

In [ ]:
import shutil
EXPORT=WORK.parent/"lecture02_advanced_submission"
EXPORT.mkdir(exist_ok=True)
for filename in ["train_messages.jsonl","validation_messages.jsonl","test_cases.json","split.json",
                 "advanced_result.json","eval_before.json","eval_after.json","eval_rag.json","answer_draft.md"]:
    shutil.copy2(WORK/filename,EXPORT/filename)
shutil.copytree(WORK/"adapter",EXPORT/"adapter",dirs_exist_ok=True)
zip_path=shutil.make_archive(str(WORK.parent/"lecture02_advanced_results"),"zip",str(EXPORT))
print(zip_path)
from google.colab import files
files.download(zip_path)

## 9. 監査済みの比較 — 検索・生成・検証を分離

元の `RAG_SYSTEM` による32回答がすべてJSONスキーマに適合しなかったことを踏まえ、厳密な形式の例を示した**新しいプロンプト**で同じ基盤モデルを再実行します。新しい生成の生スコアは新規実測として記録します。

検索で見つかった定義を**コードが直接返す**場合の結果も測ります。この直接参照ベースラインはモデル生成ではありません。モデルの生回答は、スキーマが合っていても**参照定義と完全一致する場合だけ**採用し、それ以外は出典にフォールバックします。したがって最終回答の正しさをLLMの成果として数えないことに注意してください。

In [ ]:
import json, csv, time, re, unicodedata

# 取得元は実行中に固定した glossary（講座の元ファイル）であり、
# test_cases.json の gold や reference を検索辞書として流用しない。
SCHEMA_EXAMPLE='{"answer":"参考文の定義そのもの","needs_review":false}'
REVISED_SYSTEM=(
    'あなたは講座用語集の検索結果だけを使う説明アシスタントです。'
    '次のユーザー入力にある「定義」をそのままanswerへコピーし、'
    '見つかったときはneeds_reviewをfalseにしてください。'
    '「該当なし」ならanswerを「この用語は用語集に見つかりませんでした。確認が必要です。」、'
    'needs_reviewをtrueにしてください。'
    'JSONオブジェクト1個のみ返します。キーはanswerとneeds_reviewの2つのみ。'
    'needs_reviewはtrue/falseの真偽値（文字列ではない）。'
    '例えば '+SCHEMA_EXAMPLE
)
FALLBACK='この用語は用語集に見つかりませんでした。確認が必要です。'

def grounded_answer(question):
    hit=retrieve(question)
    return ({'answer':hit['definition'],'needs_review':False},hit['id']) if hit else ({'answer':FALLBACK,'needs_review':True},None)

def generate_revised_rag(question):
    hit=retrieve(question)
    source='該当なし' if hit is None else '用語: '+hit['term']+'\n定義: '+hit['definition']
    messages=[{'role':'system','content':REVISED_SYSTEM},
              {'role':'user','content':'用語集検索結果:\n'+source+'\n\n質問: '+question}]
    prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    encoded=tok(prompt,return_tensors='pt').to(tuned.device)
    # 学習後モデルにはLoRAアダプタが組み込まれるので、必ず無効化して学習前重みと比較する。
    with tuned.disable_adapter():
        with torch.no_grad():
            output=tuned.generate(**encoded,max_new_tokens=CONFIG['max_new_tokens'],do_sample=False,
                                  pad_token_id=tok.pad_token_id)
    return tok.decode(output[0][encoded['input_ids'].shape[1]:],skip_special_tokens=True).strip()

revised_rows=[]
t0=time.perf_counter()
for case in examples:
    source,source_id=grounded_answer(case['question'])
    raw=generate_revised_rag(case['question'])
    raw_ok,raw_reason,raw_obj=check_format(raw)
    # 生成された形式の適合性と知識内容の信頼性を別々に評価する。
    # 出典定義への完全一致以外はモデルの生成文を使わない（再現性のある安全なサービス）。
    exact_source=bool(raw_ok and raw_obj==source)
    final=raw_obj if exact_source else source
    final_ok,_,_=check_format(json.dumps(final,ensure_ascii=False))
    expected_id=case['term_id']
    retrieved_ok=(source_id==expected_id)
    revised_rows.append({'category':case['category'],'question':case['question'],
        'expected_id':expected_id,'retrieved_id':source_id,'retrieval_correct':retrieved_ok,
        'llm_raw':raw,'llm_json_valid':raw_ok,'llm_json_failure_reason':raw_reason,
        'llm_matches_source':exact_source,'fallback_used':not exact_source,
        'served':final,'served_json_valid':final_ok,
        'served_definition_equals_reference':(final['answer']==case['reference'] if case['reference'] is not None else final['needs_review']),
    })
assert len(revised_rows)==32
assert all(r['retrieval_correct'] for r in revised_rows)
assert all(r['served_json_valid'] for r in revised_rows)
assert all(r['served_definition_equals_reference'] for r in revised_rows)
revision_summary={'note':'NEW Colab experiment; these numbers are generated only by running this cell',
  'n':len(revised_rows),'elapsed_seconds':round(time.perf_counter()-t0,2),
  'retrieval_id_correct':sum(r['retrieval_correct'] for r in revised_rows),
  'new_llm_json_valid':sum(r['llm_json_valid'] for r in revised_rows),
  'new_llm_source_exact':sum(r['llm_matches_source'] for r in revised_rows),
  'fallback_used':sum(r['fallback_used'] for r in revised_rows),
  'served_json_valid':sum(r['served_json_valid'] for r in revised_rows),
  'served_grounded_or_abstained':sum(r['served_definition_equals_reference'] for r in revised_rows),
  'mechanism':'source retrieval + generation + strict verifier + deterministic fallback',
  'warning':'served_... score is system-level guaranteed by source copying; not an LLM score'}
(WORK/'revised_rag_evaluation.json').write_text(json.dumps({'summary':revision_summary,'rows':revised_rows},ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(revision_summary,ensure_ascii=False,indent=2))

## 10. 新たな未見の自然文質問 — 厳密一致検索の限界を計測

これは**追加の設計確認**です。以下の質問は用語名を「」で示さないため、現在の厳密一致検索では検索できない想定です。元の32問とは独立の評価セットで、学習に使用していません。

新たな高度な検索方法（n-gram/BM25/埋め込み検索）を導入するときには、このような質問を使って、**検索結果だけ**を先に評価してください。ここでは検索範囲と限界を明示し、埋め込み検索を実装・検証したふりはしません。

In [ ]:
NATURAL_QUESTIONS=[
    ('事前学習の重みを固定し、小さな行列の積だけを学習する手法は？','lora'),
    ('LLMがツールを呼び出し、実行結果をモデルに返す処理の繰り返しは？','agent-loop'),
    ('外部サービスを呼び出すとき、チャットやGitに貼ってはいけない秘密文字列は？','api-key'),
    ('モデルが一度の生成で読み書きできるトークン数の上限は？','context-window'),
    ('作業ブランチの変更をメインに取り込むためにGitHubでレビュー依頼する仕組みは？','pull-request'),
    ('LLMが実在しない事柄をもっともらしく断言してしまう現象は？','hallucination'),
    ('未知の外部文書がモデルに無視すべき命令を埋め込み、振る舞いを変える攻撃には何の対策が必要？','guardrail'),
    ('学習データ全体を1周することは何と数える？','epoch'),
]
novel_results=[{'question':q,'gold_id':tid,'retrieved_id':retrieve(q)['id'] if retrieve(q) else None}
              for q,tid in NATURAL_QUESTIONS]
print('自然文質問:',len(novel_results),'件｜厳密一致検索で正解した件数:',sum(x['gold_id']==x['retrieved_id'] for x in novel_results))
(WORK/'natural_questions_retrieval.json').write_text(json.dumps(novel_results,ensure_ascii=False,indent=2),encoding='utf-8')


## 11. 新規再評価結果のダウンロード

実験ログと新しいRAG生出力を保存し、他の人が同じ結果を検証できるようにします。個人情報やAPIキーは保存しません。

In [ ]:
import zipfile
paths=['advanced_result.json','eval_before.json','eval_after.json','eval_rag.json',
       'test_cases.json','split.json','revised_rag_evaluation.json',
       'natural_questions_retrieval.json']
new_zip=WORK/'lecture02_revised_experiment.zip'
with zipfile.ZipFile(new_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in paths:
        path=WORK/name
        assert path.is_file(),f'ファイルが見つかりません: {path}'
        z.write(path,arcname=name)
print('保存先:',new_zip)
from google.colab import files
files.download(str(new_zip))